# 06c — 第六讲基础习题：晶体结构、XRD、凸包与组成描述符

本套习题帮助你一步步巩固第六讲（`06a`–`06b`）中 pymatgen 晶体结构建模、对称性分析、模拟 XRD、形成能与热力学凸包、Goldschmidt 容差因子以及组成特征向量（CBFV）的核心操作。每道题只需填写 **3–8 行代码**，并附有预期数值提示供你自查。完成后可对照 `solutions/06_exercises_solutions.ipynb` 查看完整解答。

---

## 0. 公共环境准备

In [ ]:
import os
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial import ConvexHull

warnings.filterwarnings("ignore")

from pymatgen.core import Structure, Lattice, Composition, Element, Species
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer
from pymatgen.analysis.diffraction.xrd import XRDCalculator

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score

plt.rcParams["figure.dpi"] = 110
np.random.seed(0)

def find_file(name):
    for path in (f"data/{name}", f"../data/{name}", name):
        if os.path.exists(path):
            return path
    raise FileNotFoundError(f"找不到 {name}")

xrd = XRDCalculator(wavelength="CuKa")
print("环境就绪，Cu K-alpha 波长 =", xrd.wavelength, "A")

<!-- ai4m-style -->
> **📘 课程统一绘图风格**：下面这格加载课程自带的 `ai4m.mplstyle`（配色与课程网站同源，无上右边框，仅横向浅网格，保存默认 300 dpi）。`C` 是课程配色，手动指定颜色时写 `color=C.crimson`。


In [ ]:
# ai4m-style：课程统一绘图风格（本地与 Colab 通用）
import pathlib, sys

sys.path.insert(0, str(pathlib.Path.cwd().parent))   # course/ 根目录
import ai4m_style as ai4m

C = ai4m.use()        # 应用 ai4m.mplstyle，返回课程配色


---

## 第一部分：晶体结构与对称性（对应 `06a`）

### 🏋️ 习题 1：搭建闪锌矿结构的 GaAs 并读出基本信息

GaAs 是闪锌矿（zinc blende）结构，空间群 `F-43m`（No. 216），晶胞参数 $a = 5.653$ Å，Ga 在 $(0, 0, 0)$，As 在 $(\tfrac{1}{4}, \tfrac{1}{4}, \tfrac{1}{4})$。

1. 用 `Structure.from_spacegroup` 搭出来；
2. 打印化学式、晶胞原子数、体积、密度；
3. 算出最近的 Ga–As 键长——先找出 Ga 和 As 的位点编号，再**对所有 As 取最小值**。周期性边界下编号相邻不等于空间相邻，随手挑两个编号相减是 `06a` 里演示过的经典错误。

- **预期输出**：8 个原子；密度约 `5.32` g/cm³；Ga–As 键长约 `2.448` Å（$= \sqrt{3}\,a/4$）。

In [ ]:
# 在下方填写你的代码：


### 🏋️ 习题 2：对称性分析与原始胞

延续上一题的 GaAs 结构：

1. 用 `SpacegroupAnalyzer(gaas, symprec=0.01)` 分析，打印空间群符号、空间群编号、晶系、点群、对称操作数；
2. 用 `get_primitive_standard_structure()` 得到原始胞，打印它的原子数和体积；
3. 计算“常规胞原子数 / 原始胞原子数”这个比值，并按原子数三次方估算：如果改用原始胞做 DFT，计算量大约能降到常规胞的百分之几。

- **预期输出**：`F-43m`（No. 216），立方晶系，点群 $\bar{4}3m$；原始胞 2 个原子；计算量约降到 $(2/8)^3 \approx 1.6\%$。

In [ ]:
# 在下方填写你的代码：


### 🏋️ 习题 3：用 Goldschmidt 容差因子判断钙钛矿能不能成

计算三个卤化物钙钛矿的容差因子

$$t = \frac{r_A + r_X}{\sqrt{2}\,(r_B + r_X)}$$

用下面这组 Shannon 离子半径（单位 Å，已按合适的配位数给出）：

| 离子 | $r$ / Å | 离子 | $r$ / Å |
|---|---|---|---|
| Cs$^+$ (XII) | 1.88 | Pb$^{2+}$ (VI) | 1.19 |
| Rb$^+$ (XII) | 1.72 | I$^-$ (VI) | 2.20 |
| K$^+$ (XII) | 1.64 | | |

1. 对 $\text{CsPbI}_3$、$\text{RbPbI}_3$、$\text{KPbI}_3$ 算出 $t$，整理成一张表；
2. 判断每一个是否落在 $0.8 < t < 1.0$ 的钙钛矿稳定窗口内；
3. **讨论**：实验上 $\text{CsPbI}_3$ 能形成钙钛矿（尽管室温下会相变成非钙钛矿的 $\delta$ 相），而 $\text{KPbI}_3$ 完全不行。可是你会发现三个 $t$ **全都落在 $0.8 \sim 1.0$ 的窗口里**，容差因子并没有把 $\text{KPbI}_3$ 排除掉。结合 `06b` 第 5 节的结论想一想：这说明容差因子的分辨力有多高？只拿它做高通量筛选，主要会产生哪一类错误——假阳性还是假阴性？

- **预期输出**：三个 $t$ 依次约为 `0.851`、`0.818`、`0.801`——一路下滑并**紧贴窗口下边界 0.8**，但谁也没掉出去。

In [ ]:
# 在下方填写你的代码：


---

## 第二部分：XRD 与热力学（对应 `06b`）

### 🏋️ 习题 4：模拟 XRD 并用布拉格定律反算 d 值

对习题 1 的 GaAs：

1. 用 `xrd.get_pattern(gaas, two_theta_range=(20, 80))` 算出衍射谱；
2. 打印前 5 个峰的 $2\theta$、$d_{hkl}$、相对强度和 $(hkl)$ 指标；
3. 取最强峰，用布拉格定律 $d = \lambda / (2\sin\theta)$ **手工反算** $d$，与 `pattern.d_hkls` 给出的值对比（注意 $\theta = 2\theta / 2$，波长用 `xrd.wavelength`）；
4. 用 `plt.vlines` 画出谱图。

- **预期输出**：最强峰是 $(111)$，位于 $2\theta \approx 27.3^\circ$；手工反算的 $d$ 与 pymatgen 的值在小数点后三位一致。

In [ ]:
# 在下方填写你的代码：


### 🏋️ 习题 5：体心立方的系统消光

体心立方（bcc）点阵的消光规律是：只有 $h + k + l$ 为**偶数**的衍射峰才出现。

1. 用 `Structure.from_spacegroup("Im-3m", Lattice.cubic(2.866), ["Fe"], [[0, 0, 0]])` 搭出 $\alpha$-Fe（bcc，$a = 2.866$ Å）；
2. 算出它的 XRD 谱（$2\theta$ 取 $20^\circ \sim 120^\circ$），把每个峰的 $(hkl)$ 和 $h+k+l$ 打印成一张表；
3. 验证列表里**所有**峰的 $h+k+l$ 都是偶数；
4. **思考**：如果某个样品的谱图里出现了 $(100)$ 峰（$h+k+l = 1$，奇数），它就不可能是单质 bcc 铁。那它可能是什么？（提示：想想有序合金，比如 CsCl 型的 FeAl）

- **预期输出**：最强峰是 $(110)$，位于 $2\theta \approx 44.7^\circ$；所有出现的 $(hkl)$ 都满足 $h+k+l$ 为偶数。

In [ ]:
# 在下方填写你的代码：


### 🏋️ 习题 6：计算形成能（含元素参考态的坑）

下面这一格读入 AFLOW 的每原子总能数据并解析化学式。请你：

1. 对每个元素，在它的所有同素异形体条目里，**先用中位数剔除偏离超过 2 eV/atom 的异常条目，再取最小值**作为参考态 $\mu$（不能直接取 `min`，`06b` 第 4 节讲过为什么）；
2. 实现 $E_f = E - \sum_i x_i \mu_i$（$x_i$ 是归一化的原子分数）；
3. 打印被判为异常的条目，以及 $\text{TiO}_2$、$\text{Al}_2\text{O}_3$、ZnO、CdTe 四个化合物的形成能。

- **预期输出**：异常条目是 `S8` 和 `Ta8`；四个形成能依次约为 $-3.16$、$-3.02$、$-1.80$、$-0.49$ eV/atom。

In [ ]:
ea = pd.read_csv(find_file("energy_per_atom_aflow.csv"))
TOKEN = re.compile(r"([A-Z][a-z]?)(\d*\.?\d*)")

def parse_formula(f):
    d = {}
    for el, n in TOKEN.findall(f):
        if el:
            d[el] = d.get(el, 0.0) + (float(n) if n else 1.0)
    return d

ea["comp"] = ea.formula.map(parse_formula)
ea["n_el"] = ea.comp.map(len)
print(f"读入 {len(ea)} 条记录，其中单质条目 {(ea.n_el == 1).sum()} 条")

# 在下方填写你的代码：


### 🏋️ 习题 7：Cu–O 体系的凸包

延续上一题算好的 `ea["E_f"]`：

1. 取出所有只含 Cu 和/或 O 的条目，算出每条的氧原子分数 $x_{\text{O}}$；
2. 对每个组成只保留能量最低的多晶型，用 `scipy.spatial.ConvexHull` 求下凸包（技巧见 `06b` 第 4 节：补一个远在上方的辅助点）；
3. 打印落在凸包上的稳定相及其 $E_f$；
4. 画出凸包图（灰点 = 全部条目，红线 = 凸包）。

- **自查**：实验上 $\text{Cu}_2\text{O}$（赤铜矿）和 CuO（黑铜矿）都是稳定存在的物相。你的凸包把它们都放上去了吗？如果没有，差了多少 meV/atom？

In [ ]:
# 在下方填写你的代码：


### 🏋️ 习题 8：CBFV 特征与一次诚实的外推测试

1. 读入 `magpie_element_properties.csv` 和 `bandgap_aflow.csv`；
2. 实现 `composition_features(comp)`：对 22 个 Magpie 属性各算 5 个统计量（加权平均、加权绝对偏差、最小值、最大值、极差），得到 110 维向量；
3. 为了课堂速度，**随机抽取 6,000 个化合物**（`sample(6000, random_state=0)`）建特征矩阵；
4. 训练 `RandomForestRegressor(n_estimators=100, random_state=0, n_jobs=-1)`，分别用两种划分报告测试集 $R^2$：
   - **随机划分**：`train_test_split(test_size=0.2, random_state=42)`；
   - **留出整类化学**：把所有**含氯（Cl）**的化合物整体作为测试集；
5. **讨论**：两个 $R^2$ 差了多少？如果你要用这个模型预测一类训练集里完全没有的新化合物，应该相信哪个数字？

- **预期输出**：随机划分的 $R^2$ 在 0.8 上下；留出氯化物后显著下降。

In [ ]:
# 在下方填写你的代码：


---

## 完成后

- 对照 `solutions/06_exercises_solutions.ipynb` 检查本套 8 道基础习题的答案；
- `06a` 与 `06b` 正文中内嵌的 6 道课堂练习（`06a-1` ~ `06a-3`、`06b-1` ~ `06b-3`）的参考解答在 `solutions/06_materials_solutions.ipynb`。